In [22]:
from typing import Union
import numpy as np
from madmom.features.downbeats import DBNDownBeatTrackingProcessor, RNNDownBeatProcessor

def detect_beats(audio: str, fps=100) -> np.ndarray:
    """
    Returns an array of shape (num_downbeats, 2) where the first column is the time of the downbeat in seconds and the second column is the beat number within the bar (starting from 1).
    """
    proc = DBNDownBeatTrackingProcessor(beats_per_bar=[3, 4], fps=fps, correct=True)
    act = RNNDownBeatProcessor()(audio)
    return proc(act)


In [11]:
detect_beats('/Users/damian/Downloads/green-day-when-i-come-around-4k-upgrade.mp3')

KeyboardInterrupt: 

In [23]:
from IPython.lib.display import Audio
import librosa
import numpy as np


def slice_downbeats(audio_path, fps=100):
    print("loading audio...")
    waveform, sampling_rate = librosa.load(audio_path)
    print("loaded")

    print("Finding beats...")
    beat_tracker_output = detect_beats(audio_path)
    downbeat_indices = np.where(beat_tracker_output[:, 1] == 1)[0]
    downbeat_times = beat_tracker_output[downbeat_indices, 0]
    print(f"Fonnd {len(downbeat_times)} downbeats at times: {downbeat_times}")

    for i in range(1, len(downbeat_indices)):
        prev_time = downbeat_times[i-1]
        this_time = downbeat_times[i]
        audio_slice = waveform[int(prev_time*sampling_rate):int(this_time*sampling_rate)]
        yield audio_slice, sampling_rate




In [24]:
slices = list(slice_downbeats('/Users/damian/2.current/talkmusic/sample2.mp3', fps=500))
#path = '/Users/damian/Downloads/green-day-when-i-come-around-4k-upgrade.mp3'


loading audio...
loaded
Finding beats...
Fonnd 5 downbeats at times: [0.26 2.14 4.01 5.88 7.75]


/Users/damian/2.current/clapSlice/.venv/lib/python3.9/site-packages/madmom/features/downbeats.py:287: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  best = np.argmax(np.asarray(results)[:, 1])


In [25]:
def display_slices(slices, offset=0, count=4):
    for i, (audio_slice, sampling_rate) in enumerate(slices):
        if i < offset:
            continue
        if i > offset+count:
            break
        print(i)

        display(Audio(audio_slice, rate=sampling_rate))


In [27]:
display_slices(slices, 0, 4)

0


1


2


3
